# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [1]:
# YOUR CODE HERE

import pandas as pd
import numpy as np
import torch
import json
import os

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
# YOUR CODE HERE

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Model loaded successfully!")


W1119 16:41:18.125000 81564 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1119 16:41:20.615000 81564 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\Users\divya\PycharmProjects\CapStoneRAGProject\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\divya\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to ru

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

C:\Users\divya\PycharmProjects\CapStoneRAGProject\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\divya\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [3]:

query = "My order is delayed. When will I receive it?"

messages = [
    {"role": "user", "content": query}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Customer Query:", query)
print("\nBaseline Model Response:", response)


Customer Query: My order is delayed. When will I receive it?

Baseline Model Response: I'm sorry to hear that your order is delayed. The exact delivery date can vary depending on several factors such as the shipping company's policies and the current inventory levels. Here are some general guidelines:

1. **Tracking Information**: Check if you have received tracking information from your online store or shipping service provider. This usually includes an estimated delivery date.

2. **Shipping Company Policies**: Different companies have different policies regarding delays. Some may offer refunds or replacements for undelivered items within a certain time frame.

3. **Inventory Levels**: If there are fewer products in stock than expected, this could


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [5]:
# YOUR CODE HERE
import json

# read the baseline answer we saved earlier
with open("outputs.json", "r", encoding="utf-8") as f:
    saved = json.load(f)

baseline_answer = saved["baseline_output"]
correct_rule = saved["ground_truth"]

print("Customer question:", saved["test_query"])
print("Correct rule:", correct_rule)
print()

text = baseline_answer.lower()

# check 1: does the answer give the real delivery time?
says_3_to_7 = "3-7" in text or "3–7" in text
print("Mentions 3-7 business days:", says_3_to_7)

# check 2: does it talk about things that are not in the company SOP?
made_up_words = ["inventory", "refund", "replacement", "shipping company"]
for word in made_up_words:
    print("Mentions", word, ":", word in text)

# check 3: was the answer cut off?
print("Cut off mid-sentence:", not baseline_answer.strip().endswith((".", "!", "?")))

Customer question: My order is delayed. When will I receive it?
Correct rule: Domestic orders deliver within 3-7 business days.

Mentions 3-7 business days: False
Mentions inventory : True
Mentions refund : True
Mentions replacement : True
Mentions shipping company : True
Cut off mid-sentence: True


##### 3.1.2 Evaluate Baseline Quality Conclusion

**Test query:** "My order is delayed. When will I receive it?"
**Correct rule (SOP):** "Domestic orders deliver within 3–7 business days."

**1. Comparison against the known rule**

The baseline model did not mention the 3–7 business day timeline specified in the SOP. Instead, it gave general suggestions about checking tracking information and said delivery dates depend on shipping company policies and inventory levels.

Verdict: the answer is factually incomplete, because the one fact the customer needed is missing.

**2. Invented timeline, tracking system or department?**

- No invented timeline: the model gave no specific date or number of days.
- No non-existent department was mentioned.
- It suggested checking tracking information from the online store or shipping provider. The SOP does not confirm this, so it is generic and unverified.

**3. Unsupported statements (hallucinations)**

Specific claims with no support in the provided documents:
- Low inventory levels may cause delivery delays (the SOP lists weather, customs and peak-season backlogs, and the order had already shipped).
- Some shipping companies may offer refunds or replacements for undelivered items within a certain time frame.

Generic statements that are plausible but unverified:
- Delivery dates vary with shipping company policies.
- Tracking information usually includes an estimated delivery date.

**4. Other quality issue**

The response was cut off in the middle of a sentence at the 120 token limit, so it is also incomplete in format.

**Learner Inference:**

The baseline produces a fluent, reasonable-sounding answer but does not give accurate, business-specific information. It missed the actual SOP rule (3–7 business days) and added general explanations that the knowledge base does not support.

This justifies Stage 3, where retrieval supplies the relevant SOP before the model answers. It also justifies Stage 4, where a fine-tuned model acts as an intent router: it extracts a structured JSON intent from a messy customer query, and that intent is used to retrieve the correct SOP. Together, retrieval and routing aim to make answers more accurate and consistent.

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [4]:
# YOUR CODE HERE

ground_truth = "Domestic orders deliver within 3-7 business days."

baseline_data = {
    "test_query": query,
    "ground_truth": ground_truth,
    "baseline_output": response
}

with open("outputs.json", "w", encoding="utf-8") as f:
    json.dump(baseline_data, f, indent=4, ensure_ascii=False)

print("Baseline output saved successfully!")


Baseline output saved successfully!


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**


##### END-OF-NOTEBOOK CHECKLIST

**IMPORTANT: All items have been verified before proceeding to Notebook 4.**

- [x] Base model (Qwen/Qwen2.5-1.5B-Instruct) loaded in 4-bit quantization without errors
- [x] Baseline output generated for the ambiguous shipping-delay test query
- [x] Baseline response evaluated against the ground-truth SOP rule
- [x] Unsupported statements and hallucination risks documented
- [x] `outputs.json` saved with `test_query`, `ground_truth` and `baseline_output`
- [x] GPU-based model inference executed successfully

**Notebook 3: Baseline Model Evaluation is complete.**



## Overall Summary

In this notebook, the base Qwen2.5-1.5B-Instruct model was loaded using 4-bit quantization to reduce memory requirements. The model was tested using an ambiguous customer query about a delayed order, without providing access to the SOP documents or performing any fine-tuning.

The model generated a response suggesting that the customer check tracking information and explaining possible reasons for delivery delays. However, it did not provide the actual delivery timeline specified in the SOP.

The response was evaluated against the ground-truth rule that domestic orders deliver within 3–7 business days. Several statements in the generated response were identified as unsupported by the provided SOP.

The test query, ground-truth rule and generated baseline response were saved in `outputs.json` for use in subsequent notebooks.
